# Сделки с жилыми помещениями в Казани: аудит качества данных

В этом ноутбуке количественно оценивается качество данных до изменения или удаления строк. Явные ошибки отделяются от необычных записей, которые требуют отдельной проверки.

In [28]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_FILE = DATA_DIR / "kazan_residential_transactions_2025_raw.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "Сначала выполните ноутбук 01_data_preparation.ipynb. "
        f"Ожидаемый файл: {INPUT_FILE}"
    )

raw = pd.read_csv(INPUT_FILE, dtype="string", encoding="utf-8")

print(f"Строк: {raw.shape[0]:,}")
print(f"Столбцов: {raw.shape[1]}")
raw.head()

Rows: 5,584
Columns: 18


,number,okato,region_code,district,city,quarter_cad_number,street,realestate_type_code,wall_material_code,year_build,floor,purpose_code,area,period_start_date,deal_price,currency,doc_type,quarter
0,1,92401380000,16,<NA>,Казань,16:50:160205,Хусаина Мавлютова,002001003000,061001001001,1986,9,206002000000,56.9,2025-01-01,8900000.0,рубль,ДКП,Q1
1,3,92401367000,16,<NA>,Казань,16:50:011102,Старая,002001003000,<NA>,<NA>,<NA>,206002000000,7991.3099999999995,2025-01-01,54923916,рубль,ДДУ,Q1
2,1,92401367000,16,<NA>,Казань,16:50:011705,Николая Столбова,002001003000,061001003000,2020,2,206002000000,62.1,2025-01-01,26400000,рубль,ДКП,Q1
3,1,92401385000,16,<NA>,Казань,16:50:060510,Комарова,002001003000,061001007001,1969,5,206002000000,59.4,2025-01-01,9000000.0,рубль,ДКП,Q1
4,1,92401363000,16,<NA>,Казань,16:50:220526,Молодежная,002001003000,061001001001,1982,5,206002000000,12.9,2025-01-01,2200000.0,рубль,ДКП,Q1


## Структура и полнота данных

In [29]:
raw.info()

coverage_summary = pd.DataFrame(
    {
        "уникальных_значений": raw.nunique(),
        "пропусков": raw.isna().sum(),
    }
)
coverage_summary = coverage_summary.sort_values(
    "уникальных_значений",
    ascending=False,
)

coverage_summary

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5584 entries, 0 to 5583
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   number                5584 non-null   string
 1   okato                 5584 non-null   string
 2   region_code           5584 non-null   string
 3   district              293 non-null    string
 4   city                  5584 non-null   string
 5   quarter_cad_number    5584 non-null   string
 6   street                5505 non-null   string
 7   realestate_type_code  5584 non-null   string
 8   wall_material_code    5272 non-null   string
 9   year_build            5269 non-null   string
 10  floor                 5283 non-null   string
 11  purpose_code          5584 non-null   string
 12  area                  5578 non-null   string
 13  period_start_date     5584 non-null   string
 14  deal_price            5584 non-null   string
 15  currency              5584 non-null   

,unique_values,missing_values
deal_price,1554,0
area,1367,6
quarter_cad_number,542,0
street,473,79
year_build,92,315
number,91,0
wall_material_code,33,312
floor,31,301
okato,8,0
period_start_date,4,0


## Пропущенные значения

Для каждого столбца рассчитываются количество и доля пропусков. Большая доля пропусков не всегда означает ошибку: некоторые поля могут быть неприменимы к отдельным видам договоров.

In [30]:
missing_summary = pd.DataFrame(
    {
        "пропущено_строк": raw.isna().sum(),
        "доля_пропусков_проц": raw.isna().mean() * 100,
    }
)

missing_summary["доля_пропусков_проц"] = missing_summary[
    "доля_пропусков_проц"
].round(2)
missing_summary = missing_summary.sort_values(
    "доля_пропусков_проц",
    ascending=False,
)
missing_summary

,missing_rows,missing_percent
district,5291,94.75
year_build,315,5.64
wall_material_code,312,5.59
floor,301,5.39
street,79,1.41
area,6,0.11
okato,0,0.00
number,0,0.00
quarter_cad_number,0,0.00
city,0,0.00


## Повторяющиеся записи

Полные дубликаты и записи с повторяющимся описанием учитываются отдельно. Из-за обезличивания одинаковые характеристики могут относиться к разным зарегистрированным договорам, поэтому такие строки не удаляются автоматически.

In [31]:
exact_duplicate_mask = raw.duplicated(keep=False)
exact_duplicates_after_first = raw.duplicated().sum()

description_columns = [
    column
    for column in raw.columns
    if column not in ["quarter", "period_start_date", "source_file"]
]
repeated_description_mask = raw.duplicated(
    subset=description_columns,
    keep=False,
)
repeated_descriptions_after_first = raw.duplicated(
    subset=description_columns
).sum()

duplicate_summary = pd.DataFrame(
    {
        "проверка": [
            "полные дубликаты после первого вхождения",
            "повторяющиеся описания после первого вхождения",
        ],
        "строк": [
            exact_duplicates_after_first,
            repeated_descriptions_after_first,
        ],
    }
)
duplicate_summary["доля_проц"] = (
    duplicate_summary["строк"] / len(raw) * 100
).round(2)
duplicate_summary

,check,rows,percent
0,exact duplicate rows after the first occurrence,0,0.00
1,repeated descriptions after the first occurrence,13,0.23


In [32]:
repeated_records = raw.loc[
    repeated_description_mask,
    [
        "number",
        "okato",
        "street",
        "year_build",
        "floor",
        "area",
        "deal_price",
        "doc_type",
        "quarter",
    ],
]
repeated_records = repeated_records.sort_values(
    ["street", "area", "deal_price"]
)
repeated_records.head(20)

,number,okato,street,year_build,floor,area,deal_price,doc_type,quarter
1342,1,92401385000,Агрызская,1978,3,17.2,2600000.0,ДКП,Q1
2523,1,92401385000,Агрызская,1978,3,17.2,2600000.0,ДКП,Q2
2405,1,92401385000,Академика Кирпичникова,1959,3,50.3,5000000.0,ДКП,Q2
5270,1,92401385000,Академика Кирпичникова,1959,3,50.3,5000000.0,ДКП,Q4
1870,1,92401370000,Большая,<NA>,<NA>,90.63,28550718,ДДУ,Q2
4004,1,92401370000,Большая,<NA>,<NA>,90.63,28550718,ДДУ,Q4
2202,1,92401380000,Ватутина,1957,1,36.6,4000000.0,ДКП,Q2
3486,1,92401380000,Ватутина,1957,1,36.6,4000000.0,ДКП,Q3
628,1,92401367000,Вишневского,2021,18,47.4,11000000,ДКП,Q1
3006,1,92401367000,Вишневского,2021,18,47.4,11000000,ДКП,Q3


## Ошибки преобразования типов

До применения правил валидности подсчитываются значения, которые невозможно преобразовать в ожидаемый числовой тип или дату.

In [33]:
audit = raw.copy()
numeric_columns = ["number", "year_build", "floor", "area", "deal_price"]
parse_summary_rows = []

for column in numeric_columns:
    audit[column] = pd.to_numeric(raw[column], errors="coerce")
    failed_to_parse = raw[column].notna() & audit[column].isna()

    parse_summary_rows.append(
        {
            "столбец": column,
            "непропущенных_исходных": raw[column].notna().sum(),
            "ошибок_преобразования": failed_to_parse.sum(),
        }
    )

audit["period_start_date"] = pd.to_datetime(
    raw["period_start_date"],
    errors="coerce",
)
date_parse_failures = (
    raw["period_start_date"].notna()
    & audit["period_start_date"].isna()
).sum()
parse_summary_rows.append(
    {
        "столбец": "period_start_date",
        "непропущенных_исходных": raw["period_start_date"].notna().sum(),
        "ошибок_преобразования": date_parse_failures,
    }
)

parse_summary = pd.DataFrame(parse_summary_rows)
parse_summary

,column,non_missing_raw_values,parse_failures
0,number,5584,0
1,year_build,5269,0
2,floor,5283,3
3,area,5578,0
4,deal_price,5584,0
5,period_start_date,5584,0


## Согласованность категориальных значений

In [34]:
category_columns = [
    "region_code",
    "city",
    "realestate_type_code",
    "purpose_code",
    "currency",
    "doc_type",
    "quarter",
]

category_names = {
    "region_code": "Код региона",
    "city": "Населенный пункт",
    "realestate_type_code": "Тип объекта недвижимости",
    "purpose_code": "Назначение помещения",
    "currency": "Валюта",
    "doc_type": "Тип договора",
    "quarter": "Квартал",
}

for column in category_columns:
    print(f"\n{category_names[column]} ({column})")
    print(raw[column].value_counts(dropna=False))


region_code
region_code
16    5584
Name: count, dtype: Int64

city
city
Казань          5581
город Казань       2
гараж              1
Name: count, dtype: Int64

realestate_type_code
realestate_type_code
002001003000    5584
Name: count, dtype: Int64

purpose_code
purpose_code
206002000000    5584
Name: count, dtype: Int64

currency
currency
рубль    5584
Name: count, dtype: Int64

doc_type
doc_type
ДКП    5282
ДДУ     302
Name: count, dtype: Int64

quarter
quarter
Q4    1622
Q1    1505
Q3    1355
Q2    1102
Name: count, dtype: Int64


In [35]:
text_columns = ["district", "city", "street", "currency", "doc_type"]
text_summary_rows = []

for column in text_columns:
    has_outer_spaces = (
        raw[column].notna()
        & (raw[column] != raw[column].str.strip())
    )
    text_summary_rows.append(
        {
            "столбец": column,
            "строк_с_внешними_пробелами": has_outer_spaces.sum(),
        }
    )

text_summary = pd.DataFrame(text_summary_rows)
composite_wall_rows = raw["wall_material_code"].str.contains(
    ";",
    regex=False,
    na=False,
).sum()

print(f"Строк с несколькими кодами материала стен: {composite_wall_rows}")
text_summary

Rows containing multiple wall-material codes: 43


,column,rows_with_outer_spaces
0,district,0
1,city,0
2,street,2
3,currency,0
4,doc_type,0


## Числовые распределения и экстремальные значения

Пороговые значения ниже используются только для диагностики, а не для автоматического удаления. Необычная запись может быть реальной сделкой или договором с несколькими объектами.

In [36]:
single_property_contract = audit["number"] == 1
positive_area_and_price = (
    single_property_contract
    & (audit["area"] > 0)
    & (audit["deal_price"] > 0)
)
audit["price_per_sqm"] = pd.NA
audit.loc[positive_area_and_price, "price_per_sqm"] = (
    audit.loc[positive_area_and_price, "deal_price"]
    / audit.loc[positive_area_and_price, "area"]
)
audit["price_per_sqm"] = pd.to_numeric(
    audit["price_per_sqm"],
    errors="coerce",
)

numeric_summary = audit[
    ["number", "year_build", "floor", "area", "deal_price", "price_per_sqm"]
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)
numeric_summary = numeric_summary.T
numeric_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
number,5584.0,2.989434,18.721095,1.0,1.0,1.0,1.0,1.0,1.0,3.0,54.17,956.0
year_build,5269.0,1993.08066,22.959121,1860.0,1941.68,1959.0,1973.0,1993.0,2016.0,2024.0,2025.0,2025.0
floor,5280.0,5.859848,4.422049,0.0,1.0,1.0,3.0,5.0,8.0,16.0,20.0,25.0
area,5578.0,9781.279529,300839.980071,3.4,11.6,16.4,33.4,44.8,62.8,192.56,10497.4677,17544423.0
deal_price,5584.0,31915005.743625,183869023.154655,149464.0,912162.0747,1900000.0,5000000.0,7000000.0,10000000.0,49974410.0,670308825.2175,6716957695.63001
price_per_sqm,5020.0,164198.089608,67772.829061,1792.134293,16985.943517,60059.601586,127684.124386,157736.119378,195008.865248,268302.48608,367036.712628,1219512.195122


## Проверки валидности и сегментации

Явные проблемы нарушают базовые правила валидности. Диагностические флаги отмечают допустимые, но необычные значения. Договоры с несколькими объектами выделяются в отдельный аналитический сегмент и не считаются ошибками.

In [37]:
expected_dates = pd.to_datetime(
    ["2025-01-01", "2025-04-01", "2025-07-01", "2025-10-01"]
)

definite_issue_masks = {
    "отсутствует или не преобразуется площадь": audit["area"].isna(),
    "площадь не является положительной": audit["area"].notna() & (audit["area"] <= 0),
    "отсутствует или не преобразуется цена договора": audit["deal_price"].isna(),
    "цена договора не является положительной": (
        audit["deal_price"].notna() & (audit["deal_price"] <= 0)
    ),
    "год постройки позднее 2025 года": (
        audit["year_build"].notna() & (audit["year_build"] > 2025)
    ),
    "этаж отрицательный": audit["floor"].notna() & (audit["floor"] < 0),
    "количество объектов не является положительным": (
        audit["number"].notna() & (audit["number"] <= 0)
    ),
    "отсутствует или не преобразуется количество объектов": audit["number"].isna(),
    "количество объектов не является целым": (
        audit["number"].notna() & (audit["number"] % 1 != 0)
    ),
    "неожиданная дата регистрации": (
        audit["period_start_date"].isna()
        | ~audit["period_start_date"].isin(expected_dates)
    ),
    "ОКАТО за пределами Казани": ~raw["okato"].str.startswith(
        "92401",
        na=False,
    ),
    "неверный или пропущенный тип недвижимости": (
        raw["realestate_type_code"].isna()
        | (raw["realestate_type_code"] != "002001003000")
    ),
    "неверное или пропущенное назначение помещения": (
        raw["purpose_code"].isna()
        | (raw["purpose_code"] != "206002000000")
    ),
    "неожиданная или пропущенная валюта": (
        raw["currency"].isna() | (raw["currency"] != "рубль")
    ),
    "неожиданный или пропущенный тип договора": (
        raw["doc_type"].isna()
        | ~raw["doc_type"].isin(["ДКП", "ДДУ"])
    ),
}

review_flag_masks = {
    "площадь одного объекта менее 10 кв. м": (
        single_property_contract
        & audit["area"].notna()
        & (audit["area"] < 10)
    ),
    "площадь одного объекта более 300 кв. м": (
        single_property_contract
        & audit["area"].notna()
        & (audit["area"] > 300)
    ),
    "цена договора с одним объектом менее 500 000 руб.": (
        single_property_contract
        & audit["deal_price"].notna()
        & (audit["deal_price"] < 500_000)
    ),
    "цена договора с одним объектом более 100 млн руб.": (
        single_property_contract
        & audit["deal_price"].notna()
        & (audit["deal_price"] > 100_000_000)
    ),
    "цена кв. м менее 30 000 руб.": (
        audit["price_per_sqm"].notna() & (audit["price_per_sqm"] < 30_000)
    ),
    "цена кв. м более 500 000 руб.": (
        audit["price_per_sqm"].notna() & (audit["price_per_sqm"] > 500_000)
    ),
    "год постройки ранее 1850 года": (
        audit["year_build"].notna() & (audit["year_build"] < 1850)
    ),
    "этаж равен 0": audit["floor"].notna() & (audit["floor"] == 0),
    "этаж выше 30": audit["floor"].notna() & (audit["floor"] > 30),
}

segment_flag_masks = {
    "договор включает несколько объектов": (
        audit["number"].notna() & (audit["number"] > 1)
    ),
}

quality_rows = []
for issue, mask in definite_issue_masks.items():
    quality_rows.append(
        {
            "категория": "явная проблема",
            "проверка": issue,
            "строк": mask.fillna(False).sum(),
        }
    )

for issue, mask in review_flag_masks.items():
    quality_rows.append(
        {
            "категория": "требует проверки",
            "проверка": issue,
            "строк": mask.fillna(False).sum(),
        }
    )

for issue, mask in segment_flag_masks.items():
    quality_rows.append(
        {
            "категория": "отдельный сегмент",
            "проверка": issue,
            "строк": mask.fillna(False).sum(),
        }
    )

quality_summary = pd.DataFrame(quality_rows)
quality_summary["доля_проц"] = (
    quality_summary["строк"] / len(audit) * 100
).round(2)
quality_summary = quality_summary.sort_values(
    ["категория", "строк"],
    ascending=[True, False],
)
quality_summary

,severity,check,rows,percent
0,definite issue,missing or unparseable area,6,0.11
1,definite issue,area is not positive,0,0.00
2,definite issue,missing or unparseable deal price,0,0.00
3,definite issue,deal price is not positive,0,0.00
4,definite issue,building year is after 2025,0,0.00
5,definite issue,floor is negative,0,0.00
6,definite issue,number is not positive,0,0.00
7,definite issue,missing or unparseable number,0,0.00
8,definite issue,number is not an integer,0,0.00
9,definite issue,unexpected registration date,0,0.00


## Качество данных по типу договора

Каждая строка соответствует одному договору. Столбец `number` содержит количество объектов недвижимости в договоре, а `deal_price` — общую цену договора. ДКП и ДДУ сравниваются отдельно, поскольку размеры договоров и полнота характеристик значительно различаются. Договоры с одним объектом можно использовать как приближение к обычным покупкам домохозяйств, а многообъектные договоры — как приближение к оптовой или институциональной активности. Это только аналитические прокси: тип покупателя в наборе данных отсутствует.

In [38]:
audit["is_multi_property_contract"] = audit["number"].notna() & (audit["number"] > 1)
audit["year_missing"] = audit["year_build"].isna()
audit["floor_missing"] = audit["floor"].isna()
audit["wall_material_missing"] = audit["wall_material_code"].isna()
audit["area_missing"] = audit["area"].isna()

contract_groups = audit.groupby("doc_type")
contract_quality = contract_groups.agg(
        contracts=("doc_type", "size"),
        property_objects=("number", "sum"),
        multi_property_contracts=("is_multi_property_contract", "sum"),
        maximum_objects_in_contract=("number", "max"),
        missing_year=("year_missing", "sum"),
        missing_floor=("floor_missing", "sum"),
        missing_wall_material=("wall_material_missing", "sum"),
        missing_area=("area_missing", "sum"),
)
contract_quality["objects_per_contract"] = (
    contract_quality["property_objects"] / contract_quality["contracts"]
).round(2)
contract_quality = contract_quality.rename(
    columns={
        "contracts": "договоров",
        "property_objects": "объектов_недвижимости",
        "multi_property_contracts": "многообъектных_договоров",
        "maximum_objects_in_contract": "максимум_объектов_в_договоре",
        "missing_year": "пропусков_года",
        "missing_floor": "пропусков_этажа",
        "missing_wall_material": "пропусков_материала_стен",
        "missing_area": "пропусков_площади",
        "objects_per_contract": "объектов_на_договор",
    }
)
contract_quality

,contracts,property_objects,multi_property_contracts,maximum_objects_in_contract,missing_year,missing_floor,missing_wall_material,missing_area,objects_per_contract
doc_type,,,,,,,,,
ДДУ,302,11050,270,956,298,298,298,6,36.59
ДКП,5282,5643,292,7,17,6,14,0,1.07


In [39]:
definite_issue_rows = pd.Series(False, index=audit.index)
for mask in definite_issue_masks.values():
    definite_issue_rows = definite_issue_rows | mask.fillna(False)

review_flag_rows = pd.Series(False, index=audit.index)
for mask in review_flag_masks.values():
    review_flag_rows = review_flag_rows | mask.fillna(False)

separate_segment_rows = pd.Series(False, index=audit.index)
for mask in segment_flag_masks.values():
    separate_segment_rows = separate_segment_rows | mask.fillna(False)

overall_quality = pd.DataFrame(
    {
        "категория": [
            "все строки",
            "строки хотя бы с одной явной проблемой",
            "строки хотя бы с одним диагностическим флагом",
            "многообъектные договоры для отдельного анализа",
            "полные дубликаты после первого вхождения",
        ],
        "строк": [
            len(audit),
            definite_issue_rows.sum(),
            review_flag_rows.sum(),
            separate_segment_rows.sum(),
            exact_duplicates_after_first,
        ],
    }
)
overall_quality["доля_проц"] = (
    overall_quality["строк"] / len(audit) * 100
).round(2)
overall_quality

,category,rows,percent
0,all rows,5584,100.00
1,rows with at least one definite issue,6,0.11
2,rows with at least one review flag,147,2.63
3,multi-property contracts outside individual-ap...,562,10.06
4,exact duplicates after the first occurrence,0,0.00


In [40]:
review_columns = [
    "number",
    "okato",
    "street",
    "year_build",
    "floor",
    "area",
    "deal_price",
    "price_per_sqm",
    "doc_type",
    "quarter",
]

flagged_records = audit.loc[
    definite_issue_rows | review_flag_rows | separate_segment_rows,
    review_columns,
]
flagged_records = flagged_records.sort_values("price_per_sqm")
flagged_records.head(30)

,number,okato,street,year_build,floor,area,deal_price,price_per_sqm,doc_type,quarter
2362,1,92401385000,Натана Рахлина,2015,8,83.4,149464.0,1792.134293,ДКП,Q2
1920,1,92401370000,Сулеймановой,2004,2,104.7,250000.0,2387.774594,ДКП,Q2
796,1,92401385000,Николая Ершова,<NA>,<NA>,10362.0,25682452.0,2478.522679,ДДУ,Q1
709,1,92401380000,Академика Парина,1992,4,114.9,300000.0,2610.966057,ДКП,Q1
1215,1,92401385000,Дорожный (Малые Клыки),1993,2,51.7,150000.0,2901.353965,ДКП,Q1
2257,1,92401380000,Кул Гали,1998,8,74.6,250000.0,3351.206434,ДКП,Q2
3105,1,92401370000,Болотникова,1964,5,54.6,209914.0,3844.578755,ДКП,Q3
1295,1,92401363000,Дементьева,1978,2,61.6,350000.0,5681.818182,ДКП,Q1
3530,1,92401380000,Дубравная,2004,5,87.9,550000.0,6257.110353,ДКП,Q3
1428,1,92401363000,Олега Кошевого,1968,2,44.2,300000.0,6787.330317,ДКП,Q1
